<a href="https://colab.research.google.com/github/pujisubarkah/replikasi_DAiSEE-Towards-User-Engagement-Recognition-in-the-Wild/blob/main/notebook_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from google.colab import drive

In [ ]:
drive.mount('/content/drive')

In [ ]:
FEATURE_ROOT = "/content/drive/MyDrive/DAiSEE_Features"

TRAIN_FEATURE = os.path.join(FEATURE_ROOT,"train")
VAL_FEATURE = os.path.join(FEATURE_ROOT,"validation")
TEST_FEATURE = os.path.join(FEATURE_ROOT,"test")

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "olgaparfenova/daisee"
)

LABEL_PATH = os.path.join(
    dataset_path,
    "DAiSEE",
    "Labels"
)

train_df = pd.read_csv(
    os.path.join(LABEL_PATH,"TrainLabels.csv")
)

val_df = pd.read_csv(
    os.path.join(LABEL_PATH,"ValidationLabels.csv")
)

test_df = pd.read_csv(
    os.path.join(LABEL_PATH,"TestLabels.csv")
)

In [ ]:
class FeatureDataset(Dataset):

    def __init__(self,
                 dataframe,
                 feature_path,
                 label_column="Engagement"):

        self.df = dataframe.reset_index(drop=True)

        self.feature_path = feature_path

        self.label_column = label_column

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        clip_name = row["ClipID"]

        clip_id = os.path.splitext(clip_name)[0]

        feature = np.load(
            os.path.join(
                self.feature_path,
                clip_id + ".npy"
            )
        )

        feature = torch.tensor(
            feature,
            dtype=torch.float32
        )

        label = torch.tensor(
            row[self.label_column],
            dtype=torch.long
        )

        return feature,label

In [ ]:
train_dataset = FeatureDataset(
    train_df,
    TRAIN_FEATURE
)

val_dataset = FeatureDataset(
    val_df,
    VAL_FEATURE
)

test_dataset = FeatureDataset(
    test_df,
    TEST_FEATURE
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
feature,label = train_dataset[0]

print(feature.shape)

print(label)

In [ ]:
class LSTMClassifier(nn.Module):

    def __init__(self,
                 input_size=2048,
                 hidden_size=256,
                 num_layers=1,
                 num_classes=4):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.5)

        self.fc = nn.Linear(
            hidden_size,
            num_classes
        )

    def forward(self,x):

        output,_ = self.lstm(x)

        output = output[:,-1,:]

        output = self.dropout(output)

        output = self.fc(output)

        return output

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = LSTMClassifier().to(device)

print(model)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [ ]:
features,labels = next(iter(train_loader))

features = features.to(device)

output = model(features)

print(output.shape)

In [ ]:
num_epochs = 10

best_val_acc = 0.0